# Ejercicio 3 — Consultas directas sobre archivos Parquet

Exploración inicial de los datos de **2026** (taxis amarillos y verdes) consultando los archivos Parquet con DuckDB, **sin importarlos a una tabla**.

* Las consultas están en [`sql/03_exploracion.sql`](../sql/03_exploracion.sql). Cada una tiene un comentario con su **objetivo** y los **archivos fuente**; aquí se imprime la consulta, se muestra el **resultado** y se anota la **decisión** tomada (3.8).
* La conexión es en memoria (`conectar_parquet()` en [`scripts/lab.py`](../scripts/lab.py)): solo define las vistas `viajes` ([`sql/00_vistas.sql`](../sql/00_vistas.sql)) y `viajes_validos` ([`sql/01_limpieza.sql`](../sql/01_limpieza.sql)), que son consultas guardadas sobre `read_parquet`, no copias de los datos.

In [1]:
import sys

import pandas as pd

sys.path.append("../scripts")
from lab import conectar_parquet, consultas, mostrar

pd.set_option("display.max_columns", None)
con = conectar_parquet()
Q = consultas("03_exploracion.sql")

## 3.1 Cantidad de archivos disponibles

In [2]:
mostrar(con, Q["3.1_archivos"])

-- Objetivo: cantidad de archivos Parquet disponibles por tipo de taxi y meses cubiertos.
-- Fuente: nombres de archivo de data/raw/*/2026/*.parquet (glob, no lee datos).
SELECT regexp_extract(file, 'raw/(\w+)/', 1)                    AS tipo_taxi,
       count(*)                                                 AS archivos,
       min(regexp_extract(file, '(\d{4}-\d{2})\.parquet$', 1))  AS primer_mes,
       max(regexp_extract(file, '(\d{4}-\d{2})\.parquet$', 1))  AS ultimo_mes
FROM glob('data/raw/*/2026/*.parquet')
GROUP BY tipo_taxi
ORDER BY tipo_taxi;


,tipo_taxi,archivos,primer_mes,ultimo_mes
0,green,8,2026-01,2026-08
1,yellow,8,2026-01,2026-08


**Resultado:** 16 archivos, 8 por tipo de taxi, de 2026-01 a 2026-08.

**Decisión:** el año está incompleto porque la TLC publica con semanas de atraso (septiembre a diciembre aún no existen). Todo resultado de 2026 se refiere a enero–agosto y las comparaciones con otros años deben hacerse sobre los mismos meses.

## 3.2 Cantidad de registros

In [3]:
mostrar(con, Q["3.2_registros_por_archivo"])

-- Objetivo: registros de cada archivo, leidos del pie (metadatos) del Parquet sin escanear los datos.
-- Fuente: data/raw/*/2026/*.parquet (parquet_file_metadata).
SELECT regexp_extract(file_name, '(\w+_tripdata_\d{4}-\d{2})', 1) AS archivo,
       num_rows                                                 AS registros,
       num_row_groups                                           AS row_groups
FROM parquet_file_metadata('data/raw/*/2026/*.parquet')
ORDER BY archivo;


,archivo,registros,row_groups
0,green_tripdata_2026-01,40272,1
1,green_tripdata_2026-02,37373,1
2,green_tripdata_2026-03,44208,1
3,green_tripdata_2026-04,44238,1
4,green_tripdata_2026-05,44921,1
5,green_tripdata_2026-06,44163,1
6,green_tripdata_2026-07,41252,1
7,green_tripdata_2026-08,40687,1
8,yellow_tripdata_2026-01,3724889,4
9,yellow_tripdata_2026-02,3399866,4


In [4]:
mostrar(con, Q["3.2_registros_total"])

-- Objetivo: total de registros por tipo de taxi contando las filas de los archivos.
-- Fuente: data/raw/yellow/2026/*.parquet y data/raw/green/2026/*.parquet.
SELECT 'yellow' AS tipo_taxi, count(*) AS registros
FROM read_parquet('data/raw/yellow/2026/*.parquet', union_by_name = true)
UNION ALL
SELECT 'green', count(*)
FROM read_parquet('data/raw/green/2026/*.parquet', union_by_name = true);


,tipo_taxi,registros
0,yellow,29703355
1,green,337114


**Resultado:** 29,703,355 registros de amarillos y 337,114 de verdes (30,040,469 en total). La suma de `num_rows` de los metadatos coincide con el conteo de filas, y cada mes tiene entre 3.3 y 4.1 millones de viajes amarillos y entre 37 y 45 mil verdes.

**Decisión:** los verdes son ~1% del total, así que comparar totales entre tipos no es informativo; se comparan proporciones, promedios y percentiles. Contar con `parquet_file_metadata` es instantáneo porque solo lee el pie de cada archivo.

## 3.3 Columnas presentes en los archivos

In [5]:
mostrar(con, Q["3.3_columnas_yellow"])

-- Objetivo: columnas y tipos de datos de los archivos de taxis amarillos.
-- Fuente: data/raw/yellow/2026/*.parquet.
DESCRIBE SELECT * FROM read_parquet('data/raw/yellow/2026/*.parquet', union_by_name = true);


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [6]:
mostrar(con, Q["3.3_columnas_green"])

-- Objetivo: columnas y tipos de datos de los archivos de taxis verdes.
-- Fuente: data/raw/green/2026/*.parquet.
DESCRIBE SELECT * FROM read_parquet('data/raw/green/2026/*.parquet', union_by_name = true);


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,lpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,lpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,store_and_fwd_flag,VARCHAR,YES,None,None,None
4,RatecodeID,BIGINT,YES,None,None,None
5,PULocationID,INTEGER,YES,None,None,None
6,DOLocationID,INTEGER,YES,None,None,None
7,passenger_count,BIGINT,YES,None,None,None
8,trip_distance,DOUBLE,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None


**Resultado:** amarillos tienen 21 columnas y verdes 22. Comparten la mayoría (proveedor, zonas de origen/destino `PULocationID`/`DOLocationID`, distancia, tarifa y recargos, forma de pago), pero:

* las fechas se llaman `tpep_*` en amarillos y `lpep_*` en verdes;
* `Airport_fee` solo existe en amarillos; `ehail_fee` y `trip_type` (1 = parada en la calle, 2 = despachado) solo en verdes;
* `cbd_congestion_fee` (cargo por congestión de Manhattan, vigente desde enero de 2025) está en ambos.

**Decisión:** para analizar ambos tipos juntos, la vista `viajes` renombra las fechas a `pickup_datetime` / `dropoff_datetime` y une los tipos por nombre de columna (`UNION ALL BY NAME`); las columnas exclusivas de un tipo quedan en NULL para el otro.

## 3.4 Tipos de datos de las columnas

Los tipos se ven en la columna `column_type` de las dos consultas anteriores. La siguiente consulta revisa el esquema **de cada archivo** para detectar columnas que no están en todos o cuyo tipo cambia entre meses:

In [7]:
mostrar(con, Q["3.4_esquema_por_archivo"])

-- Objetivo: detectar columnas que no estan en todos los archivos o cuyo tipo fisico cambia entre archivos.
-- Fuente: esquema de cada archivo de data/raw/*/2026/*.parquet (parquet_schema).
WITH esquema AS (
    SELECT regexp_extract(file_name, 'raw/(\w+)/', 1)                AS tipo_taxi,
           regexp_extract(file_name, '(\d{4}-\d{2})\.parquet$', 1)   AS mes,
           name                                                      AS columna,
           type                                                      AS tipo_fisico
    FROM parquet_schema('data/raw/*/2026/*.parquet')
    WHERE num_children IS NULL      -- excluye el nodo raiz del esquema
)
SELECT tipo_taxi,
       columna,
       count(*)                            AS archivos_con_columna,
       count(DISTINCT tipo_fisico)         AS tipos_fisicos,
       min(mes)                            AS desde
FROM esquema
GROUP BY tipo_taxi, columna
QUALIFY archivos_con_columna < max(archivos_con_columna) OVER (PARTITION BY tipo_tax

,tipo_taxi,columna,archivos_con_columna,tipos_fisicos,desde
0,green,request_source,3,1,2026-06
1,yellow,request_source,3,1,2026-06


**Resultado:**

* fechas: `TIMESTAMP` (sin zona horaria, hora local de NY);
* códigos categóricos guardados como enteros: `VendorID`, `PULocationID`, `DOLocationID` (`INTEGER`), `RatecodeID`, `payment_type`, `passenger_count`, `trip_type` (`BIGINT`);
* distancia y montos: `DOUBLE`; `store_and_fwd_flag` y `request_source`: `VARCHAR`;
* ningún tipo cambia entre archivos de 2026, pero **`request_source` aparece desde 2026-06** (3 de 8 archivos por tipo).

**Decisión:** todas las lecturas usan `union_by_name = true`. Sin esa opción DuckDB toma el esquema del primer archivo y la columna nueva se pierde; con ella, los meses sin la columna la tienen en NULL. Los códigos numéricos se tratan como categorías (no se promedian).

## 3.5 Muestra de registros

In [8]:
mostrar(con, Q["3.5_muestra_yellow"])

-- Objetivo: muestra aleatoria de registros de taxis amarillos.
-- Fuente: data/raw/yellow/2026/*.parquet.
SELECT *
FROM read_parquet('data/raw/yellow/2026/*.parquet', union_by_name = true)
USING SAMPLE reservoir(5 ROWS) REPEATABLE (42);


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,1,2026-01-01 00:34:45,2026-01-01 00:40:15,4,0.90,1,N,262,141,1,7.9,3.50,0.5,3.20,0.00,1.0,16.10,2.5,0.0,0.00,None
1,1,2026-01-01 00:06:07,2026-01-01 00:20:20,2,1.90,1,N,231,90,1,14.2,4.25,0.5,3.95,0.00,1.0,23.90,2.5,0.0,0.75,None
2,2,2026-01-01 03:15:42,2026-01-01 03:20:33,1,1.15,1,N,263,140,1,7.2,1.00,0.5,2.44,0.00,1.0,14.64,2.5,0.0,0.00,None
3,2,2026-01-01 04:25:35,2026-01-01 04:38:18,1,3.69,1,N,107,143,1,17.7,1.00,0.5,4.69,0.00,1.0,28.14,2.5,0.0,0.75,None
4,2,2026-01-01 06:33:56,2026-01-01 06:55:37,2,10.27,2,N,68,138,2,70.0,0.00,0.5,0.00,6.94,1.0,81.69,2.5,0.0,0.75,None


In [9]:
mostrar(con, Q["3.5_muestra_green"])

-- Objetivo: muestra aleatoria de registros de taxis verdes.
-- Fuente: data/raw/green/2026/*.parquet.
SELECT *
FROM read_parquet('data/raw/green/2026/*.parquet', union_by_name = true)
USING SAMPLE reservoir(5 ROWS) REPEATABLE (42);


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,request_source
0,2,2026-01-02 05:43:34,2026-01-02 05:44:09,N,5,34,34,2,0.40,70.0,0.0,0.0,14.20,0.0,NaN,1.0,85.20,1,2,0.0,0.0,None
1,2,2026-01-02 21:02:00,2026-01-02 21:07:42,N,1,74,41,1,1.17,7.9,1.0,0.5,2.08,0.0,NaN,1.0,12.48,1,1,0.0,0.0,None
2,1,2026-01-14 21:03:03,2026-01-14 21:06:16,N,1,181,65,1,0.60,5.8,1.0,1.5,0.00,0.0,NaN,1.0,8.30,2,1,0.0,0.0,None
3,2,2026-01-16 19:11:20,2026-01-16 19:29:44,N,1,74,151,1,2.33,17.7,2.5,0.5,4.34,0.0,NaN,1.0,26.04,1,1,0.0,0.0,None
4,2,2026-01-19 02:15:45,2026-01-19 02:20:57,N,1,95,102,1,1.35,8.6,1.0,0.5,1.00,0.0,NaN,1.0,12.10,1,1,0.0,0.0,None


**Resultado:** cada registro es un viaje con fechas de inicio y fin, zonas, distancia en millas y el desglose del cobro (`fare_amount` + recargos + propina = `total_amount`).

**Decisión:** la muestra usa `USING SAMPLE reservoir(...)` en vez de `LIMIT`, para no ver solo las primeras filas del primer archivo (que están ordenadas por fecha).

## 3.6 Problemas de calidad de datos

### Valores nulos (% por columna)

In [10]:
mostrar(con, Q["3.6_nulos"]).set_index("tipo_taxi").T

-- Objetivo: porcentaje de valores nulos por columna y tipo de taxi.
-- Fuente: vista `viajes` (sql/00_vistas.sql), es decir, los Parquet de 2026.
SELECT tipo_taxi,
       round(100.0 * count_if(COLUMNS(* EXCLUDE (tipo_taxi, anio, mes)) IS NULL) / count(*), 2)
FROM viajes
WHERE anio = 2026
GROUP BY tipo_taxi
ORDER BY tipo_taxi;


tipo_taxi,green,yellow
VendorID,0.00,0.00
pickup_datetime,0.00,0.00
dropoff_datetime,0.00,0.00
passenger_count,14.47,25.98
trip_distance,0.00,0.00
RatecodeID,14.47,25.98
store_and_fwd_flag,14.47,25.98
PULocationID,0.00,0.00
DOLocationID,0.00,0.00
payment_type,14.47,0.00


El 100% de nulos en `Airport_fee` (verdes) y `trip_type` (amarillos) es esperado: esas columnas no existen en ese tipo. Los demás nulos sí son un problema: `passenger_count`, `RatecodeID`, `store_and_fwd_flag` y `congestion_surcharge` faltan en **25.98%** de los amarillos y **14.47%** de los verdes, siempre en los mismos registros. `ehail_fee` está vacío en todos los verdes y `request_source` solo existe desde junio.

In [11]:
mostrar(con, Q["3.6_incompletos_por_pago"])

-- Objetivo: verificar si los nulos se concentran en un grupo de registros (codigo de pago).
-- Fuente: vista `viajes`, anio 2026.
SELECT tipo_taxi,
       payment_type,
       count(*)                                   AS registros,
       count(*) - count(passenger_count)          AS sin_passenger_count,
       count(*) - count(RatecodeID)               AS sin_ratecode,
       count(*) - count(congestion_surcharge)     AS sin_congestion_surcharge,
       round(avg(tip_amount), 2)                  AS propina_promedio
FROM viajes
WHERE anio = 2026
GROUP BY ALL
ORDER BY tipo_taxi, payment_type;


,tipo_taxi,payment_type,registros,sin_passenger_count,sin_ratecode,sin_congestion_surcharge,propina_promedio
0,green,1,219980,0,0,0,3.84
1,green,2,65921,0,0,0,0.00
2,green,3,1688,0,0,0,0.00
3,green,4,750,0,0,0,0.00
4,green,<NA>,48775,48775,48775,48775,0.79
5,yellow,0,7716688,7716688,7716688,7716688,0.40
6,yellow,1,18941008,0,0,0,4.28
7,yellow,2,2708031,0,0,0,0.00
8,yellow,3,98138,0,0,0,0.00
9,yellow,4,239488,0,0,0,0.00


Los registros incompletos son exactamente los de `payment_type = 0` en amarillos (7.7 millones) y `payment_type` NULL en verdes. Según el diccionario de datos de la TLC, 0 corresponde a viajes *flex fare*; además son los que reportan la propina menos confiable (USD 0.40 de promedio frente a 4.28 con tarjeta).

### Rangos y valores extremos

In [12]:
mostrar(con, Q["3.6_rangos"]).set_index("tipo_taxi").T

-- Objetivo: rangos y percentiles extremos de fechas, distancias, duraciones y montos.
-- Fuente: vista `viajes`, anio 2026.
SELECT tipo_taxi,
       min(pickup_datetime)                                                AS pickup_min,
       max(pickup_datetime)                                                AS pickup_max,
       min(trip_distance)                                                  AS distancia_min,
       quantile_cont(trip_distance, 0.5)                                   AS distancia_p50,
       quantile_cont(trip_distance, 0.999)                                 AS distancia_p999,
       max(trip_distance)                                                  AS distancia_max,
       quantile_cont(date_diff('second', pickup_datetime, dropoff_datetime) / 60, 0.999)
                                                                           AS duracion_min_p999,
       min(fare_amount)                                                    AS tarifa_min,
       quantile_cont(fare_am

tipo_taxi,green,yellow
pickup_min,2008-12-31 17:35:31,2001-01-01 09:23:58
pickup_max,2026-08-31 23:58:28,2026-08-31 23:59:59
distancia_min,0.0,0.0
distancia_p50,2.07,1.86
distancia_p999,30.05,30.73646
distancia_max,179830.92,328522.2
duracion_min_p999,1405.922033,118.460767
tarifa_min,-500.0,-2555.2
tarifa_p50,13.5,15.6
tarifa_max,1676.7,7045.0


In [13]:
mostrar(con, Q["3.6_problemas"]).set_index("tipo_taxi").T

-- Objetivo: cuantificar cada problema de calidad detectado.
-- Fuente: vista `viajes`, anio 2026.
SELECT tipo_taxi,
       count(*)                                                                      AS registros,
       count_if(year(pickup_datetime) <> anio OR month(pickup_datetime) <> mes)::BIGINT AS fecha_fuera_del_mes,
       count_if(dropoff_datetime <= pickup_datetime)::BIGINT                         AS duracion_no_positiva,
       count_if(dropoff_datetime > pickup_datetime + INTERVAL 3 HOUR)::BIGINT        AS duracion_mayor_3h,
       count_if(trip_distance = 0)::BIGINT                                           AS distancia_cero,
       count_if(trip_distance > 100)::BIGINT                                         AS distancia_mayor_100mi,
       count_if(fare_amount < 0 OR total_amount < 0)::BIGINT                         AS montos_negativos,
       count_if(fare_amount = 0)::BIGINT                                             AS tarifa_cero,
       count_if(passenger_count =

tipo_taxi,green,yellow
registros,337114,29703355
fecha_fuera_del_mes,98,146
duracion_no_positiva,234,371683
duracion_mayor_3h,1287,10127
distancia_cero,12212,952231
distancia_mayor_100mi,72,1223
montos_negativos,1025,161970
tarifa_cero,4880,22387
pasajeros_cero,4527,91359
zona_desconocida,5886,201486


**Problemas encontrados en 2026:**

| Problema | Amarillos | Verdes |
|---|---:|---:|
| Fecha de inicio fuera del mes del archivo (hasta 2001 y 2008) | 146 | 98 |
| Duración cero o negativa | 371,683 | 234 |
| Duración mayor a 3 horas (p99.9 de verdes: ~23 h) | 10,127 | 1,287 |
| Distancia 0 | 952,231 | 12,212 |
| Distancia mayor a 100 millas (máximo: 328,522 mi) | 1,223 | 72 |
| Montos negativos (tarifa mínima: −2,555.20) | 161,970 | 1,025 |
| Tarifa 0 | 22,387 | 4,880 |
| `passenger_count` = 0 | 91,359 | 4,527 |
| Zona desconocida o fuera de NYC (264/265) | 201,486 | 5,886 |
| Registros con nulos en 5 columnas (`payment_type` 0/NULL) | 7,716,688 | 48,775 |
| Columna nueva a mitad de año (`request_source`) | desde 2026-06 | desde 2026-06 |

**Decisión:** se creó la vista [`viajes_validos`](../sql/01_limpieza.sql), que excluye registros con fecha fuera del mes del archivo, duración ≤ 0 o > 3 h, distancia ≤ 0 o > 100 millas, tarifa o total ≤ 0 y velocidad promedio > 80 mph. Los registros con `passenger_count` nulo **no** se excluyen (son un cuarto de los datos y el resto del viaje es válido); los promedios de pasajeros se calculan solo sobre los registros que lo informan. Las zonas 264/265 se conservan porque el viaje en sí es válido.

In [14]:
mostrar(con, Q["3.6_efecto_limpieza"])

-- Objetivo: registros que conserva la vista viajes_validos (sql/01_limpieza.sql).
-- Fuente: vistas `viajes` y `viajes_validos`, anio 2026.
SELECT v.tipo_taxi,
       v.registros,
       l.validos,
       round(100.0 * l.validos / v.registros, 2) AS pct_validos
FROM (SELECT tipo_taxi, count(*) AS registros FROM viajes WHERE anio = 2026 GROUP BY ALL) v
JOIN (SELECT tipo_taxi, count(*) AS validos FROM viajes_validos WHERE anio = 2026 GROUP BY ALL) l
  USING (tipo_taxi)
ORDER BY v.tipo_taxi;


,tipo_taxi,registros,validos,pct_validos
0,green,337114,316783,93.97
1,yellow,29703355,28212241,94.98


La limpieza conserva 94.98% de los amarillos y 93.97% de los verdes de 2026.

## 3.7 Consultas directamente sobre Parquet

Todas las consultas de este notebook leen los archivos en el momento de ejecutarse: `glob()` lista los archivos, `parquet_file_metadata()` / `parquet_schema()` leen solo los metadatos, `read_parquet()` lee los datos, y las vistas `viajes` / `viajes_validos` son consultas guardadas sobre `read_parquet()`. No se ejecutó ningún `CREATE TABLE` ni se cargó nada a pandas salvo los resultados ya agregados.

## 3.8 Documentación de las consultas

| Consulta (`sql/03_exploracion.sql`) | Objetivo | Fuente | Resultado | Decisión |
|---|---|---|---|---|
| `3.1_archivos` | Archivos por tipo | nombres de `data/raw/*/2026/*.parquet` | 8 + 8 archivos, 2026-01 a 2026-08 | 2026 se analiza como enero–agosto |
| `3.2_registros_por_archivo` | Registros por archivo | metadatos de `data/raw/*/2026/*.parquet` | 3.3–4.1 M (amarillos), 37–45 mil (verdes) por mes | contar con metadatos es instantáneo |
| `3.2_registros_total` | Total de registros | `data/raw/{yellow,green}/2026/*.parquet` | 29,703,355 + 337,114 | comparar proporciones, no totales |
| `3.3_columnas_yellow` / `_green` | Columnas y tipos | `data/raw/{yellow,green}/2026/*.parquet` | 21 y 22 columnas; fechas `tpep_`/`lpep_` | vista unificada `viajes` |
| `3.4_esquema_por_archivo` | Cambios de esquema entre archivos | esquema de cada `data/raw/*/2026/*.parquet` | `request_source` desde 2026-06 | `union_by_name = true` |
| `3.5_muestra_yellow` / `_green` | Ver registros reales | `data/raw/{yellow,green}/2026/*.parquet` | 5 viajes por tipo | muestreo aleatorio, no `LIMIT` |
| `3.6_nulos` | % de nulos | vista `viajes` (2026) | 25.98% / 14.47% en 5 columnas | no excluir por `passenger_count` |
| `3.6_incompletos_por_pago` | Origen de los nulos | vista `viajes` (2026) | nulos = `payment_type` 0 / NULL | se etiqueta como "Flex fare / sin dato" |
| `3.6_rangos` / `3.6_problemas` | Valores imposibles | vista `viajes` (2026) | ver tabla de 3.6 | reglas de `viajes_validos` |
| `3.6_efecto_limpieza` | Registros que conserva la limpieza | vistas `viajes` y `viajes_validos` (2026) | 94.98% / 93.97% | se usa `viajes_validos` en el análisis |

## 3.9 ¿Qué significa consultar directamente un archivo Parquet?

Significa que DuckDB ejecuta el SQL leyendo el archivo en su formato original, sin un paso previo de carga a una base de datos. Funciona bien porque Parquet es un formato **columnar** con **metadatos**:

* **Lectura por columnas:** una consulta que usa 3 de 20 columnas solo lee esas 3 del disco.
* **Metadatos en el pie:** número de filas, esquema y mínimo/máximo de cada columna por *row group*. Con ellos DuckDB responde un `count(*)` sin leer datos y descarta *row groups* completos cuando un filtro no puede cumplirse (*predicate pushdown*).
* **Compresión:** los 30 millones de viajes de 2026 ocupan unos 495 MiB en Parquet; leer menos bytes compensa el costo de descomprimir.
* **Sin duplicar datos ni pasos de carga:** el archivo descargado es la fuente; agregar un mes es agregar un archivo que el glob ya incluye.
* **Fuera de memoria:** DuckDB procesa los archivos por bloques y en paralelo, así que el volumen no está limitado por la RAM (a diferencia de cargar todo en pandas).

Con volúmenes grandes esto evita mantener una copia de los datos en otra base y permite explorar los archivos nuevos en cuanto se descargan.